In [ ]:
# ==============================================================================
# ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================

# Install dependencies if running in a cloud notebook (e.g., Google Colab).
# !pip install -q transformers

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW
from sklearn.metrics import mean_absolute_error, cohen_kappa_score
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# ==============================================================================
# WESTERN CORPUS INGESTION & DATALOADER PREPARATION
# ==============================================================================
# NOTE FOR REVIEWERS: Set `base_path` to the repository root directory.
base_path = '..'

print("1. Loading Western ASAP Dataset (Prompt 1)...")
asap_df = pd.read_csv(f'{base_path}/data/asap_dataset_prompt_1.csv')

# Min-Max Scaling (ASAP Prompt 1 targets strictly range from 2.0 to 12.0)
asap_df['normalized_score'] = (asap_df['domain1_score'] - 2.0) / 10.0

print("2. Initializing BERT Tokenizer & PyTorch Dataset...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class StandardASAPDataset(Dataset):
    def __init__(self, texts, targets, tokenizer_obj, max_len=512):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer_obj
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'target': torch.tensor(self.targets[idx], dtype=torch.float)
        }

# 80/20 Holdout Split for Native Evaluation
train_df, val_df = train_test_split(asap_df, test_size=0.2, random_state=42)

train_dataset = StandardASAPDataset(train_df['essay'].values, train_df['normalized_score'].values, tokenizer)
val_dataset = StandardASAPDataset(val_df['essay'].values, val_df['normalized_score'].values, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
print(f"DataLoaders established. Training samples: {len(train_dataset)} | Validation samples: {len(val_dataset)}")

1. Loading Western ASAP Dataset (Prompt 1)...
2. Initializing BERT Tokenizer & PyTorch Dataset...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

DataLoaders established. Training samples: 1426 | Validation samples: 357


In [ ]:
# ==============================================================================
# ARCHITECTURE DEFINITION & BASE TRAINING
# ==============================================================================
class StandardAESModel(nn.Module):
    def __init__(self):
        super(StandardAESModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.drop = nn.Dropout(p=0.3)
        self.regressor = nn.Linear(768, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = self.drop(outputs.pooler_output)
        raw_score = self.regressor(pooled_output)
        return self.sigmoid(raw_score)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StandardAESModel().to(device)
print(f"Neural Architecture instantiated on {device}")

EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.MSELoss()

print("\nInitiating Western Baseline Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['target'].to(device)

        preds = model(input_ids, attention_mask).squeeze()
        loss = loss_fn(preds, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"[Epoch {epoch+1}/{EPOCHS}] Mean Squared Error: {total_loss/len(train_loader):.4f}")

# Serialize the 'Teacher' weights for Warm-Start transfer learning
artifact_dir = f'{base_path}/artifacts/03_Experiment_A_Western_Baseline'
os.makedirs(artifact_dir, exist_ok=True)
torch.save(model.state_dict(), f'{artifact_dir}/western_asap_teacher.pth')
print(f"\nTraining Complete. Full Teacher Model serialized to:\n{artifact_dir}/western_asap_teacher.pth")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Neural Architecture instantiated on cuda

Initiating Western Baseline Training...
[Epoch 1/3] Mean Squared Error: 0.0157
[Epoch 2/3] Mean Squared Error: 0.0093
[Epoch 3/3] Mean Squared Error: 0.0072

Training Complete. Full Teacher Model serialized to:
/content/drive/MyDrive/AES_Files/Artifacts/03_Experiment_A_Western_Baseline/western_asap_teacher.pth


In [5]:
# ==============================================================================
# NATIVE DOMAIN EVALUATION (ASAP VALIDATION)
# ==============================================================================
def asap_continuous_to_discrete(scores):
    # Reverse Min-Max mapping to project continuous [0.0, 1.0] back to ASAP [2, 12] scale
    discrete_scores = np.round((scores * 10.0) + 2.0)
    return np.clip(discrete_scores, 2, 12).astype(int)

model.eval()
val_preds, val_targets = [], []

print("Executing Inference on ASAP Validation Holdout...")
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['target'].numpy()

        preds = model(input_ids, attention_mask).squeeze().cpu().numpy()
        val_preds.extend(preds.flatten())
        val_targets.extend(targets.flatten())

val_preds = np.array(val_preds)
val_targets = np.array(val_targets)

qwk = cohen_kappa_score(
    asap_continuous_to_discrete(val_targets),
    asap_continuous_to_discrete(val_preds),
    weights='quadratic'
)
print(f"Native ASAP QWK (Sanity Check): {qwk:.4f}")

Executing Inference on ASAP Validation Holdout...
Native ASAP QWK (Sanity Check): 0.8374


In [ ]:
# ==============================================================================
# ZERO-SHOT CROSS-DOMAIN EVALUATION (COVARIATE SHIFT AUDIT)
# ==============================================================================
print("Initiating Zero-Shot Covariate Shift Audit on BSSD...")

# 1. Load Localized BSSD Artifact
bssd_df = pd.read_csv(f'{base_path}/data/final_preprocessed_data.csv')
text_col = 'clean_essay_text' if 'clean_essay_text' in bssd_df.columns else 'clean_text'
bssd_texts = bssd_df[text_col].tolist()
bssd_targets = bssd_df['final_target_score'].values

raw_asap_preds = []
model.eval()

# 2. Execute Zero-Shot Inference
with torch.no_grad():
    for text in bssd_texts:
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512, padding='max_length').to(device)
        pred = model(inputs['input_ids'], inputs['attention_mask']).squeeze().item()
        raw_asap_preds.append(pred)

# 3. Calculate Zero-Shot QWK (0-10 Discretization)
# Matches the mathematical scaling used across the rest of the research framework
actuals_binned = np.round(bssd_targets * 10).astype(int)
preds_binned = np.round(np.array(raw_asap_preds) * 10).astype(int)

qwk = cohen_kappa_score(actuals_binned, preds_binned, weights='quadratic')

print("\n==================================================")
print("CROSS-DOMAIN COVARIATE SHIFT RESULTS")
print("==================================================")
print(f"Zero-Shot Baseline QWK on Bangladeshi Data: {qwk:.4f}")

# 4. Export Artifact
out_df = pd.DataFrame({'asap_zero_shot_preds': raw_asap_preds})
out_path = f'{artifact_dir}/asap_predictions.csv'
out_df.to_csv(out_path, index=False)
print(f"\nZero-Shot continuous predictions successfully archived to:\n{out_path}")

Initiating Zero-Shot Covariate Shift Audit on BSSD...

CROSS-DOMAIN COVARIATE SHIFT RESULTS
Zero-Shot Baseline QWK on Bangladeshi Data: 0.6189

Zero-Shot continuous predictions successfully archived to:
/content/drive/MyDrive/AES_Files/Artifacts/03_Experiment_A_Western_Baseline/asap_predictions.csv
